### ch3 Functional data structures

In [1]:
sealed class List<out A> {
    companion object {
        operator fun <A> invoke(vararg aa: A): List<A> =
            if (aa.isEmpty())
                Nil
            else
                Cons(aa[0], List(*aa.sliceArray(1 .. aa.size - 1)))

        fun <A> empty(): List<A> =
            Nil
    }
}

object Nil: List<Nothing>() {
    override fun toString(): String = "Nil"
}

data class Cons<out A>(val head: A, val tail: List<A>) : List<A>() {
    override fun toString(): String =
        "$head :: ${tail.toString()}"
}

In [2]:
val ex1 = Nil
val ex2 = Cons(1, Nil)
val ex3 = Cons("a", Cons("b", Nil))

//val cons = List(1, 2)
//val nil = List<Int>()

In [3]:
fun sum(ints: List<Int>): Int =
    when (ints) {
        is Nil -> 0
        is Cons -> ints.head + sum(ints.tail)
        else -> error("")
    }

fun product(doubles: List<Double>): Double {
    tailrec fun loop(doubles: List<Double>, acc: Double): Double =
        when (doubles) {
            is Nil -> acc
            is Cons ->
                if (doubles.head == 0.0) 0.0
                else loop(doubles.tail, doubles.head * acc)
            else -> error("")
        }
    return loop(doubles, 1.0)
}

product(List(1.0, 2.0, 3.0))


6.0

In [4]:
import kotlin.random.Random

val x = Random.nextInt(-10, 10)
val y = when {
    x == 0 -> "x is zero"
    x < 0 -> "x is negative"
    else -> "x is positive"
}
y

x is negative

#### ex 3.1

In [5]:
fun <A> tail(xs: List<A>): List<A> =
    when (xs) {
        is Nil -> error("Nil")
        is Cons -> xs.tail
        else -> error("")
    }

#### ex 3.2

In [6]:
fun <A> setHead(xs: List<A>, x: A): List<A> =
    when (xs) {
        is Nil -> error("Nil")
        is Cons -> Cons(x, xs.tail)
        else -> error("")
    }

setHead(List(1, 2, 3), 10)

10 :: 2 :: 3 :: Nil

#### ex 3.3

In [7]:
tailrec fun <A> drop(xs: List<A>, n: Int): List<A> =
    if (n == 0)
        xs
    else
        when (xs) {
            is Nil -> error("")
            is Cons -> drop(xs.tail, n - 1)
            else -> error("")
        }

drop(List(1, 2, 3), 2)

3 :: Nil

#### 3.4

In [8]:
fun <A> dropWhile(l: List<A>, f: (A) -> Boolean): List<A> =
    when (l) {
        is Nil -> Nil
        is Cons -> if (f(l.head)) dropWhile(l.tail, f) else l
        else -> error("")
    }

dropWhile(List(1, 3, 1, 2, 3)) { it % 2 == 1 }

2 :: 3 :: Nil

In [9]:
fun <A> append(a1: List<A>, a2: List<A>): List<A> =
    when (a1) {
        is Nil -> a2
        is Cons -> Cons(a1.head, append(a1.tail, a2))
        else -> error("")
    }

#### ex 3.5

In [10]:
fun <A> init(l: List<A>): List<A> =
    when (l) {
        is Nil -> error("Nil")
        is Cons -> when (l.tail) {
            is Nil -> Nil
            is Cons -> Cons(l.head, init(l.tail))
            else -> error("")
        }
        else -> error("")
    }


init(List(1, 2, 3))


1 :: 2 :: Nil

In [11]:
fun <A, B> foldRight(xs: List<A>, z: B, f: (A, B) -> B): B =
    when (xs) {
        is Nil -> z
        is Cons -> f(xs.head, foldRight(xs.tail, z, f))
        else -> error("")
    }
foldRight(List(1, 2, 3), 0) { a, b -> a + b }
// (1 + (2 + (3 + 0)))


//foldRight(List(3, 6, 9), 1.0) { a, b -> a * b }
// foldRight(List(1, 2, 3), 0) { a, b -> a - b }


6

#### ex 3.6

In [12]:
// 모든 식이 확장되어 하나씩 평가되므로 중간에 빠져 나오는건 불가능
// f 에서 foldRight 를 빠져나올 수 있는 goto 가 필요하다. 없음

#### ex 3.7

In [13]:
// list 를 복사한다

#### ex 3.8

In [14]:
fun <A> length(xs: List<A>): Int =
    foldRight(xs, 0) { _, l -> l + 1 }

length(List(1, 2, 3))

3

#### ex 3.9

In [15]:
tailrec fun <A, B> foldLeft(xs: List<A>, z: B, f: (B, A) -> B): B =
    when (xs) {
        is Nil -> z
        is Cons -> foldLeft(xs.tail, f(z, xs.head), f)
        else -> error("")
    }

// ((1 * 2) * 3) * 4
println(foldLeft(List(2, 3, 4), 1) { n, acc -> n * acc })

24


#### ex 3.10

In [16]:
fun sum(l: List<Int>): Int =
    foldLeft(l, 0) { acc, n -> acc + n }

println(sum(List(3, 4, 5)))

fun product(l: List<Double>): Double =
    foldLeft(l, 1.0) { acc, n -> acc * n }

println(product(List(4.0, 5.0, 6.0)))

12
120.0


#### ex 3.11

In [33]:
//fun <A> reverse(l: List<A>): List<A> =
//    when (l) {
//        is Nil -> Nil
//        is Cons -> append(reverse(l.tail), Cons(l.head, Nil))
//        else -> error("")
//    }

fun <A> reverse(l: List<A>): List<A> =
    foldLeft(l, List.empty()) { acc, i -> Cons(i, acc) }

println(reverse<Int>(List(1, 2, 3)))

3 :: 2 :: 1 :: Nil


#### ex 3.12 (hard)

In [59]:
fun <A, B> foldRightA(xs: List<A>, z: B, f: (A, B) -> B): B =
    foldLeft(xs, { b: B -> b }) { acc, a ->
        { b: B -> acc(f(a, b)) }
    }(z)

// foldLeft(List(1, 2, 3), { b0 -> b0 })(Nil)
// foldLeft(List(2, 3), { b1 -> { b0 -> b0 }(f(1, b1)) })(Nil)
// foldLeft(List(3), { b2 -> { b1 -> { b0 -> b0 }(f(1, b1)) }(f(2, b2)) })(Nil)
// foldLeft(Nil, {b3 -> { b2 -> { b1 -> { b0 -> b0 }(f(1, b1)) }(f(2, b2)) }(f(3, b3)) })(Nil)
// { b3 -> { b2 -> { b1 -> { b0 -> b0 }(f(1, b1)) }(f(2, b2)) }(f(3, b3)) }(Nil)
// { b2 -> { b1 -> { b0 -> b0 }(f(1, b1)) }(f(2, b2)) }(f(3, Nil))
// { b2 -> { b1 -> { b0 -> b0 }(f(1, b1)) }(f(2, b2)) }(Cons(3, Nil))
// { b1 -> { b0 -> b0 }(f(1, b1)) }(f(2, Cons(3, Nil)))
// { b1 -> { b0 -> b0 }(f(1, b1)) }(Cons(2, Cons(3, Nil)))
// { b0 -> b0 }(f(1, Cons(2, Cons(3, Nil))))
// { b0 -> b0 }(Cons(1, Cons(2, Cons(3, Nil))))
// Cons(1, Cons(2, Cons(3, Nil)))

println(foldRight(List(1, 2, 3), List.empty<Int>()) { a, acc -> Cons(a, acc) })
println(foldRightA(List(1, 2, 3), List.empty<Int>()) { a, acc -> Cons(a, acc) })

// foldLeft(List(1, 2, 3), Nil)
// { foldLeft(List(1, 2, 3), Nil) }
// Cons(1, { foldLeft(List(2, 3), Nil) } )
// Cons(1, Cons(2, { foldLeft(List(3), Nil) } ))
// Cons(1, Cons(2, Cons(3, { foldLeft(Nil, Nil) } )))

// { Cons(1, Cons(2, (foldLeft(List(3), Nil))) }
// { Cons(1, Cons(2, Cons(3, foldLeft(Nil, Nil)))) }
// { Cons(1, Cons(2, Cons(3, Nil))) }


// foldLeft(List(1, 2, 3), Nil)
// foldLeft(List(2, 3), Cons(1, Nil))
// foldLeft(List(3), Cons(1, Cons(2, Nil)))
// foldLeft(Nil, Cons(1, Cons(2, Cons(3, Nil))))


//foldRight(List(1, 2, 3, 4, 5), 0) { a, acc -> a - acc }
// 1 - (2 - (3 - (4 - 0))) = -2

//foldRightA(List(1, 2, 3, 4, 5), 0) { a, acc -> a - acc }

// foldLeft(List(4, 3, 2, 1), 0) { a, acc -> a - acc }
// ((((4 - 0) - 3) - 2) - 1) = -2

//foldLeft(List(1, 2, 3), 0) { a, acc -> a - acc }
// ((0 - 1) - 2) - 3 = -6

// (3, 2, 1)
// 3 - (2 - (1 - 0))

1 :: 2 :: 3 :: Nil
1 :: 2 :: 3 :: Nil


#### ex 3.13

In [22]:
fun <A> append1(l1: List<A>, l2: List<A>): List<A> =
    foldRight(l1, l2) { a, acc -> Cons(a, acc) }

println(append1(List(1, 2), List(3, 4)))

fun <A> append2(l1: List<A>, l2: List<A>): List<A> =
    foldLeft(reverse(l1), l2) { acc, a -> Cons(a, acc) }

println(append2(List(1, 2), List(3, 4)))

1 :: 2 :: 3 :: 4 :: Nil
1 :: 2 :: 3 :: 4 :: Nil


#### ex 3.14

In [39]:
fun <A> concat1(ll: List<List<A>>): List<A> =
    reverse(foldLeft(ll, List.empty<A>()) { acc, l -> foldLeft(l, acc) { acc, a -> Cons(a, acc) } })

fun <A> concat(ll: List<List<A>>): List<A> =
    foldRight(ll, List.empty<A>()) { l, acc -> foldRight(l, acc) { a, acc -> Cons(a, acc) } }

concat(List(List(1, 2), List(3, 4), List(5, 6)))

1 :: 2 :: 3 :: 4 :: 5 :: 6 :: Nil